# Final split comparison

Use the selected application-only 64 → 32 network with Adam weight decay 0.0001 in three evaluation protocols. **Stratified is the main development setup.** Random splitting represents the same broad applicant population without explicitly preserving TARGET proportions. The nonrandom scenario asks whether training on applicants with recorded previous Home Credit applications transfers to applicants without them.

Keep the fixed 500 eyeball applicants out of every alternate training, dev and test partition. The remaining 307,011 applicants are the common comparison universe. Main stratified results are reused from the frozen selected-model export; the two other protocols fit their own preprocessing, checkpoint and F1 threshold using only their own training/dev data. Split choice is not a hyperparameter, and scores on different test populations do not identify the best split.

All results are exploratory: earlier work exposed overlapping test records and motivated the previous-application scenario. Do not tune after viewing these results. No individual test or dev errors are displayed here. Prior application membership is not a fairness guarantee or proof of prior borrowing. The count is excluded from model inputs.

This audit uses the current gender-excluded model. Random and stratified use equal train/dev/test counts (184,506 / 61,002 / 61,503) on the same universe; previous random experiments used slightly different counts. The no-prior-application scenario keeps its natural deployment-group sizes.


In [1]:
from pathlib import Path
import copy
import random

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
from torch import nn
from torch.utils.data import DataLoader, TensorDataset

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.metrics import roc_auc_score, accuracy_score, f1_score, precision_score, recall_score

SEED = 42
EPOCHS = 15
BATCH_SIZE = 256
LEARNING_RATE = 0.001

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Running on:", device)
import json, hashlib, joblib
from datetime import datetime, timezone
from IPython.display import display
from sklearn.metrics import confusion_matrix


Running on: cpu


Matplotlib is building the font cache; this may take a moment.


In [2]:
data_path = Path("application_train.csv")
if not data_path.exists():
    data_path = Path("/content/application_train.csv")
if not data_path.exists():
    data_path = Path.home() / "Downloads" / "Project 2 DATA" / "application_train.csv"

df = pd.read_csv(data_path)
applicant_ids = df["SK_ID_CURR"].copy()
y = df["TARGET"].copy()
X = df.drop(columns=["TARGET", "SK_ID_CURR", "CODE_GENDER"]).copy()
print("Applicants:", len(df))
assert "CODE_GENDER" not in X.columns
# Keep raw df gender only for fairness auditing, never model input.

main_pointer=Path('active_error_analysis_run.json')
if not main_pointer.exists(): main_pointer=data_path.parent/'active_error_analysis_run.json'
main_dir=Path(json.loads(main_pointer.read_text())['input_dir'])
if not main_dir.is_absolute(): main_dir=main_pointer.parent/main_dir
main_info=json.loads((main_dir/'model_info.json').read_text())
assert main_info['weight_decay']==0.0001
assert 'CODE_GENDER' in main_info['excluded_predictors']
assert 'CODE_GENDER' not in X.columns
manifest=json.loads((main_dir/'dev_split_manifest.json').read_text())
eyeball_ids=set(manifest['eyeball_ids'])
eligible_indices=df.index[~df.SK_ID_CURR.isin(eyeball_ids)]
assert len(eligible_indices)==307011
X['DAYS_EMPLOYED_PLACEHOLDER']=X.DAYS_EMPLOYED.eq(365243).astype(int)
X.loc[X.DAYS_EMPLOYED.eq(365243),'DAYS_EMPLOYED']=np.nan
run_dir=Path('final_split_runs')/datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%S%fZ')
run_dir.mkdir(parents=True,exist_ok=False)
(run_dir/'protocol.json').write_text(json.dumps(dict(main_run=str(main_dir),seed=SEED,
    weight_decay=0.0001,excluded_predictors=main_info['excluded_predictors'],random_counts_matched_to_stratified=True,excluded_eyeball_ids=sorted(eyeball_ids),historical_exposure=True),indent=2))
print('Main model:', main_dir, '| Comparison universe:',len(eligible_indices))


Applicants: 307511
Main model: model_runs/application_no_gender_weight_decay_20261013T064129748622Z | Comparison universe: 307011


## 1 Shared preprocessing and model fitting

In [3]:
# Categorical columns
categorical_cols = [
    'NAME_CONTRACT_TYPE', 'FLAG_OWN_CAR',
    'FLAG_OWN_REALTY', 'NAME_TYPE_SUITE', 'NAME_INCOME_TYPE',
    'NAME_EDUCATION_TYPE', 'NAME_FAMILY_STATUS', 'NAME_HOUSING_TYPE',
    'OCCUPATION_TYPE', 'WEEKDAY_APPR_PROCESS_START', 'ORGANIZATION_TYPE',
    'FONDKAPREMONT_MODE', 'HOUSETYPE_MODE', 'WALLSMATERIAL_MODE',
    'EMERGENCYSTATE_MODE', 'REGION_RATING_CLIENT', 'REGION_RATING_CLIENT_W_CITY',
]
# Numeric measurements
scale_cols = [
    'CNT_CHILDREN', 'AMT_INCOME_TOTAL', 'AMT_CREDIT',
    'AMT_ANNUITY', 'AMT_GOODS_PRICE', 'REGION_POPULATION_RELATIVE',
    'DAYS_BIRTH', 'DAYS_EMPLOYED', 'DAYS_REGISTRATION',
    'DAYS_ID_PUBLISH', 'OWN_CAR_AGE', 'CNT_FAM_MEMBERS',
    'HOUR_APPR_PROCESS_START', 'EXT_SOURCE_1', 'EXT_SOURCE_2',
    'EXT_SOURCE_3', 'APARTMENTS_AVG', 'BASEMENTAREA_AVG',
    'YEARS_BEGINEXPLUATATION_AVG', 'YEARS_BUILD_AVG', 'COMMONAREA_AVG',
    'ELEVATORS_AVG', 'ENTRANCES_AVG', 'FLOORSMAX_AVG',
    'FLOORSMIN_AVG', 'LANDAREA_AVG', 'LIVINGAPARTMENTS_AVG',
    'LIVINGAREA_AVG', 'NONLIVINGAPARTMENTS_AVG', 'NONLIVINGAREA_AVG',
    'APARTMENTS_MODE', 'BASEMENTAREA_MODE', 'YEARS_BEGINEXPLUATATION_MODE',
    'YEARS_BUILD_MODE', 'COMMONAREA_MODE', 'ELEVATORS_MODE',
    'ENTRANCES_MODE', 'FLOORSMAX_MODE', 'FLOORSMIN_MODE',
    'LANDAREA_MODE', 'LIVINGAPARTMENTS_MODE', 'LIVINGAREA_MODE',
    'NONLIVINGAPARTMENTS_MODE', 'NONLIVINGAREA_MODE', 'APARTMENTS_MEDI',
    'BASEMENTAREA_MEDI', 'YEARS_BEGINEXPLUATATION_MEDI', 'YEARS_BUILD_MEDI',
    'COMMONAREA_MEDI', 'ELEVATORS_MEDI', 'ENTRANCES_MEDI',
    'FLOORSMAX_MEDI', 'FLOORSMIN_MEDI', 'LANDAREA_MEDI',
    'LIVINGAPARTMENTS_MEDI', 'LIVINGAREA_MEDI', 'NONLIVINGAPARTMENTS_MEDI',
    'NONLIVINGAREA_MEDI', 'TOTALAREA_MODE', 'OBS_30_CNT_SOCIAL_CIRCLE',
    'DEF_30_CNT_SOCIAL_CIRCLE', 'OBS_60_CNT_SOCIAL_CIRCLE', 'DEF_60_CNT_SOCIAL_CIRCLE',
    'DAYS_LAST_PHONE_CHANGE', 'AMT_REQ_CREDIT_BUREAU_HOUR', 'AMT_REQ_CREDIT_BUREAU_DAY',
    'AMT_REQ_CREDIT_BUREAU_WEEK', 'AMT_REQ_CREDIT_BUREAU_MON', 'AMT_REQ_CREDIT_BUREAU_QRT',
    'AMT_REQ_CREDIT_BUREAU_YEAR',
]
# Binary flags
binary_cols = [
    'FLAG_MOBIL', 'FLAG_EMP_PHONE', 'FLAG_WORK_PHONE',
    'FLAG_CONT_MOBILE', 'FLAG_PHONE', 'FLAG_EMAIL',
    'REG_REGION_NOT_LIVE_REGION', 'REG_REGION_NOT_WORK_REGION', 'LIVE_REGION_NOT_WORK_REGION',
    'REG_CITY_NOT_LIVE_CITY', 'REG_CITY_NOT_WORK_CITY', 'LIVE_CITY_NOT_WORK_CITY',
    'FLAG_DOCUMENT_2', 'FLAG_DOCUMENT_3', 'FLAG_DOCUMENT_4',
    'FLAG_DOCUMENT_5', 'FLAG_DOCUMENT_6', 'FLAG_DOCUMENT_7',
    'FLAG_DOCUMENT_8', 'FLAG_DOCUMENT_9', 'FLAG_DOCUMENT_10',
    'FLAG_DOCUMENT_11', 'FLAG_DOCUMENT_12', 'FLAG_DOCUMENT_13',
    'FLAG_DOCUMENT_14', 'FLAG_DOCUMENT_15', 'FLAG_DOCUMENT_16',
    'FLAG_DOCUMENT_17', 'FLAG_DOCUMENT_18', 'FLAG_DOCUMENT_19',
    'FLAG_DOCUMENT_20', 'FLAG_DOCUMENT_21', 'DAYS_EMPLOYED_PLACEHOLDER',
]

In [4]:
from sklearn.metrics import confusion_matrix
from IPython.display import display

def score_partition(actual, probabilities, cutoff):
    labels = probabilities >= cutoff
    tn, fp, fn, tp = confusion_matrix(actual, labels, labels=[0, 1]).ravel()
    return dict(n=len(actual), difficulty_prevalence=float(np.mean(actual)),
        majority_accuracy=max(float(np.mean(actual)), 1-float(np.mean(actual))),
        auc=roc_auc_score(actual, probabilities), accuracy=accuracy_score(actual, labels),
        precision=precision_score(actual, labels, zero_division=0),
        recall=recall_score(actual, labels, zero_division=0), f1=f1_score(actual, labels, zero_division=0),
        false_positive_rate=fp/(fp+tn), tn=int(tn), fp=int(fp), fn=int(fn), tp=int(tp))

# This code is embedded in both notebooks so no extra module upload is required.
def fairness_tables(predictions, attributes, cutoff):
    """Audit TARGET=1 difficulty; undefined rates stay NaN, never zero."""
    data = predictions.merge(attributes[['SK_ID_CURR', 'CODE_GENDER', 'DAYS_BIRTH']],
                             on='SK_ID_CURR', validate='one_to_one')
    data['age_group'] = pd.cut(-data.DAYS_BIRTH / 365.25, [0, 30, 45, 60, np.inf],
        right=False, labels=['Under 30', '30 to under 45', '45 to under 60', '60 and older'])
    data['label'] = data.predicted_probability.ge(cutoff)
    rows = []
    for attribute in ['CODE_GENDER', 'age_group']:
        for group, part in data.groupby(attribute, observed=True, dropna=False):
            actual = part.actual_TARGET.eq(1); label = part.label
            tp=int((actual & label).sum()); fn=int((actual & ~label).sum())
            fp=int((~actual & label).sum()); tn=int((~actual & ~label).sum())
            counts = {'false_positive_rate': (fp, fp+tn), 'recall': (tp, tp+fn),
                      'precision': (tp, tp+fp), 'npv': (tn, tn+fn),
                      'specificity': (tn, tn+fp), 'false_negative_rate': (fn, fn+tp),
                      'favorable_prediction_rate': (tn+fn, len(part)),
                      'difficulty_prevalence': (tp+fn, len(part))}
            row=dict(attribute=attribute, group=str(group), n=len(part),
                     positives=tp+fn, negatives=tn+fp, tp=tp, fn=fn, fp=fp, tn=tn)
            for metric, (success, total) in counts.items():
                rate=success/total if total else np.nan
                row[metric]=rate; row[metric+'_denominator']=total
                # Descriptive 95% Wilson interval; does not account for repeated dev selection.
                if total:
                    z=1.959963984540054; den=1+z*z/total
                    center=(rate+z*z/(2*total))/den
                    half=z*np.sqrt(rate*(1-rate)/total+z*z/(4*total*total))/den
                    row[metric+'_lower']=max(0.,center-half); row[metric+'_upper']=min(1.,center+half)
                else:
                    row[metric+'_lower']=np.nan; row[metric+'_upper']=np.nan
            row['small_group_warning']=min(tp+fn,tn+fp,tp+fp,tn+fn)<30
            rows.append(row)
    groups=pd.DataFrame(rows)
    gaps=[]
    for attribute in groups.attribute.unique():
        part=groups[groups.attribute.eq(attribute)]
        for metric in ['false_positive_rate','recall','precision','npv','favorable_prediction_rate','specificity']:
            eligible=part[part[metric+'_denominator'].ge(30)&part[metric].notna()]
            if len(eligible)>=2:
                lo=eligible.loc[eligible[metric].idxmin()]; hi=eligible.loc[eligible[metric].idxmax()]
                gaps.append(dict(attribute=attribute,metric=metric,eligible_groups=len(eligible),
                    excluded_groups='; '.join(part.loc[~part.index.isin(eligible.index),'group']),
                    lowest_group=lo['group'],highest_group=hi['group'],
                    minimum=lo[metric],maximum=hi[metric],gap_pp=100*(hi[metric]-lo[metric])))
    return groups, pd.DataFrame(gaps)


In [5]:
def fit_protocol(name, train_idx, dev_idx, test_idx):
    parts=[set(train_idx),set(dev_idx),set(test_idx)]
    assert not (parts[0]&parts[1] or parts[0]&parts[2] or parts[1]&parts[2])
    assert set.union(*parts)==set(eligible_indices)
    assert set(df.loc[list(set.union(*parts)), 'SK_ID_CURR']).isdisjoint(eyeball_ids)
    X_train,X_val,X_test=[X.loc[idx].copy() for idx in [train_idx,dev_idx,test_idx]]
    y_train,y_val,y_test=[y.loc[idx].copy() for idx in [train_idx,dev_idx,test_idx]]
    missing_columns = [
        column for column in scale_cols
        if column != "DAYS_EMPLOYED" and X_train[column].isna().any()
    ]

    for column in missing_columns:
        flag = column + "_MISSING"
        X_train[flag] = X_train[column].isna().astype(int)
        X_val[flag] = X_val[column].isna().astype(int)

    missing_flags = [column + "_MISSING" for column in missing_columns]
    model_binary_cols = binary_cols + missing_flags
    print("Added numeric missingness flags:", len(missing_flags))
    preprocessor = ColumnTransformer([
        ("categorical", Pipeline([
            ("impute", SimpleImputer(strategy="most_frequent")),
            ("encode", OneHotEncoder(handle_unknown="ignore"))
        ]), categorical_cols),

        ("numeric", Pipeline([
            ("impute", SimpleImputer(strategy="median")),
            ("scale", StandardScaler())
        ]), scale_cols),

        ("binary", SimpleImputer(strategy="most_frequent"), model_binary_cols)
    ], sparse_threshold=0)

    train_array = preprocessor.fit_transform(X_train).astype(np.float32)
    val_array = preprocessor.transform(X_val).astype(np.float32)
    assert np.isfinite(train_array).all() and np.isfinite(val_array).all()
    print("Training shape:", train_array.shape)
    assert not any("CODE_GENDER" in name for name in preprocessor.get_feature_names_out())

    random.seed(SEED)
    np.random.seed(SEED)
    torch.manual_seed(SEED)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(SEED)
    torch.backends.cudnn.benchmark = False
    torch.backends.cudnn.deterministic = True

    train_X_tensor = torch.from_numpy(train_array).to(device)
    val_X_tensor = torch.from_numpy(val_array).to(device)
    train_y_tensor = torch.tensor(y_train.to_numpy(), dtype=torch.float32, device=device)
    val_y_tensor = torch.tensor(y_val.to_numpy(), dtype=torch.float32, device=device)

    train_loader = DataLoader(
        TensorDataset(train_X_tensor, train_y_tensor),
        batch_size=BATCH_SIZE, shuffle=True,
        generator=torch.Generator().manual_seed(SEED)
    )

    model = nn.Sequential(
        nn.Linear(train_array.shape[1], 64),
        nn.ReLU(),
        nn.Linear(64, 32),
        nn.ReLU(),
        nn.Linear(32, 1)
    ).to(device)

    loss_fn = nn.BCEWithLogitsLoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=LEARNING_RATE, weight_decay=0.0001)
    best_auc = -np.inf
    history = []

    for epoch in range(1, EPOCHS + 1):
        model.train()
        for batch_X, batch_y in train_loader:
            optimizer.zero_grad()
            loss = loss_fn(model(batch_X).squeeze(1), batch_y)
            loss.backward()
            optimizer.step()

        model.eval()
        with torch.no_grad():
            train_logits = model(train_X_tensor).squeeze(1)
            val_logits = model(val_X_tensor).squeeze(1)
            train_prob = torch.sigmoid(train_logits).cpu().numpy()
            val_prob = torch.sigmoid(val_logits).cpu().numpy()
            train_loss = loss_fn(train_logits, train_y_tensor).item()
            val_loss = loss_fn(val_logits, val_y_tensor).item()

        train_auc = roc_auc_score(y_train, train_prob)
        val_auc = roc_auc_score(y_val, val_prob)
        history.append({"epoch": epoch, "train_loss": train_loss, "val_loss": val_loss,
                        "train_auc": train_auc, "val_auc": val_auc})
        print(f"Epoch {epoch:02d}: train AUC={train_auc:.4f}, validation AUC={val_auc:.4f}")

        if val_auc > best_auc:
            best_auc = val_auc
            best_epoch = epoch
            best_weights = copy.deepcopy(model.state_dict())

    model.load_state_dict(best_weights)
    model.eval()
    with torch.no_grad():
        val_probabilities = torch.sigmoid(model(val_X_tensor).squeeze(1)).cpu().numpy()

    print(f"Best epoch: {best_epoch}; validation AUC: {best_auc:.5f}")
    threshold_results = []
    for step in range(10, 501):
        threshold = step / 1000
        predictions = val_probabilities >= threshold
        threshold_results.append({
            "threshold": threshold,
            "f1": f1_score(y_val, predictions, zero_division=0),
            "precision": precision_score(y_val, predictions, zero_division=0),
            "recall": recall_score(y_val, predictions, zero_division=0),
            "accuracy": accuracy_score(y_val, predictions)
        })

    threshold_results = pd.DataFrame(threshold_results)
    best_row = threshold_results.loc[threshold_results["f1"].idxmax()]
    best_threshold = float(best_row["threshold"])
    print(f"F1-selected threshold: {best_threshold:.1%}")
    threshold_results.loc[
        threshold_results["threshold"].isin([0.10, 0.125, best_threshold, 0.15, 0.50])
    ].round(4)
    # Model and threshold fixed before the first test prediction.
    for column in missing_columns:
        X_test[column+'_MISSING']=X_test[column].isna().astype(int)
    test_array=preprocessor.transform(X_test).astype(np.float32)
    assert np.isfinite(test_array).all()
    def predict_array(array):
        model.eval()
        with torch.no_grad():
            return np.concatenate([torch.sigmoid(model(torch.from_numpy(array[i:i+4096]).to(device)).squeeze(1)).cpu().numpy()
                                   for i in range(0,len(array),4096)])
    folder=run_dir/name;folder.mkdir()
    rows=[]
    for partition,actual,array,idx in [('train',y_train,train_array,train_idx),('blackbox',y_val,val_array,dev_idx),('test',y_test,test_array,test_idx)]:
        probabilities=predict_array(array)
        rows.append(dict(protocol=name,partition=partition,selected_epoch=best_epoch,threshold=best_threshold,
                         **score_partition(actual,probabilities,best_threshold)))
        predictions=pd.DataFrame({'SK_ID_CURR':applicant_ids.loc[idx].to_numpy(),
            'actual_TARGET':actual.to_numpy(),'predicted_probability':probabilities})
        if partition!='train':
            predictions.to_json(folder/f'{partition}_predictions.json',orient='records',indent=2)
            groups,gaps=fairness_tables(predictions,df,best_threshold)
            groups.to_json(folder/f'{partition}_fairness.json',orient='records',indent=2)
            gaps.to_json(folder/f'{partition}_fairness_gaps.json',orient='records',indent=2)
    pd.DataFrame(rows).to_json(folder/'metrics.json',orient='records',indent=2)
    pd.DataFrame(history).to_json(folder/'learning_curves.json',orient='records',indent=2)
    joblib.dump(preprocessor,folder/'preprocessor.joblib')
    torch.save({k:v.detach().cpu() for k,v in model.state_dict().items()},folder/'weights.pt')
    (folder/'settings.json').write_text(json.dumps(dict(seed=SEED,weight_decay=0.0001,
        selected_epoch=int(best_epoch),threshold=best_threshold,main_model_selected_before_evaluation=True,
        split_ids={label:applicant_ids.loc[idx].tolist() for label,idx in [('train',train_idx),('blackbox',dev_idx),('test',test_idx)]}),indent=2))
    return pd.DataFrame(rows)


## 2 Random sampling without stratification

In [6]:
random_train,random_test=train_test_split(eligible_indices,test_size=61503,random_state=SEED)
random_train,random_dev=train_test_split(random_train,test_size=61002,random_state=SEED)
random_metrics=fit_protocol('random',random_train,random_dev,random_test)
display(random_metrics.round(4))


Added numeric missingness flags: 60
Training shape: (184506, 306)
Epoch 01: train AUC=0.7459, validation AUC=0.7430
Epoch 02: train AUC=0.7489, validation AUC=0.7447
Epoch 03: train AUC=0.7508, validation AUC=0.7455
Epoch 04: train AUC=0.7521, validation AUC=0.7463
Epoch 05: train AUC=0.7534, validation AUC=0.7461
Epoch 06: train AUC=0.7529, validation AUC=0.7455
Epoch 07: train AUC=0.7557, validation AUC=0.7464
Epoch 08: train AUC=0.7562, validation AUC=0.7470
Epoch 09: train AUC=0.7589, validation AUC=0.7459
Epoch 10: train AUC=0.7592, validation AUC=0.7467
Epoch 11: train AUC=0.7608, validation AUC=0.7466
Epoch 12: train AUC=0.7620, validation AUC=0.7471
Epoch 13: train AUC=0.7647, validation AUC=0.7481
Epoch 14: train AUC=0.7640, validation AUC=0.7470
Epoch 15: train AUC=0.7648, validation AUC=0.7491
Best epoch: 15; validation AUC: 0.74910
F1-selected threshold: 12.8%


,protocol,partition,selected_epoch,threshold,n,difficulty_prevalence,majority_accuracy,auc,accuracy,precision,recall,f1,false_positive_rate,tn,fp,fn,tp
0,random,train,15,0.128,184506,0.0810,0.9190,0.7648,0.8277,0.2305,0.4823,0.3120,0.1419,145504,24057,7737,7208
1,random,blackbox,15,0.128,61002,0.0795,0.9205,0.7491,0.8255,0.2167,0.4573,0.2941,0.1427,48137,8015,2632,2218
2,random,test,15,0.128,61503,0.0811,0.9189,0.7503,0.8211,0.2162,0.4591,0.2940,0.1470,48208,8305,2699,2291


## 3 Nonrandom holdout by previous application availability

All applicants without a recorded earlier application are test cases. Applicants with previous records are split 80/20, stratified by TARGET, into train and blackbox dev. These sizes follow the deployment population rather than a forced 60/20/20 allocation. All statuses count; only DAYS_DECISION < 0 is eligible. No applicant appears in more than one partition within this protocol.


In [7]:
paths=[data_path.parent/'previous_application.csv',Path('/content/previous_application.csv'),Path.home()/'Downloads'/'previous_application.csv']
previous_path=next((p for p in paths if p.exists()),None)
if previous_path is None: raise FileNotFoundError('Provide previous_application.csv alongside application_train.csv.')
previous=pd.read_csv(previous_path,usecols=['SK_ID_CURR','SK_ID_PREV','DAYS_DECISION'])
assert previous.SK_ID_PREV.is_unique
prior_count=previous.loc[previous.DAYS_DECISION.lt(0)].groupby('SK_ID_CURR').size()
membership=applicant_ids.map(prior_count).fillna(0).gt(0)
known=eligible_indices[membership.loc[eligible_indices].to_numpy()]
new=eligible_indices[~membership.loc[eligible_indices].to_numpy()]
prior_train,prior_dev=train_test_split(known,test_size=.20,random_state=SEED,stratify=y.loc[known])
assert membership.loc[prior_train].all() and membership.loc[prior_dev].all() and not membership.loc[new].any()
prior_metrics=fit_protocol('no_previous_application',prior_train,prior_dev,new)
display(prior_metrics.round(4))


Added numeric missingness flags: 59
Training shape: (232472, 303)
Epoch 01: train AUC=0.7447, validation AUC=0.7412
Epoch 02: train AUC=0.7452, validation AUC=0.7397
Epoch 03: train AUC=0.7470, validation AUC=0.7395
Epoch 04: train AUC=0.7487, validation AUC=0.7424
Epoch 05: train AUC=0.7495, validation AUC=0.7428
Epoch 06: train AUC=0.7511, validation AUC=0.7440
Epoch 07: train AUC=0.7530, validation AUC=0.7452
Epoch 08: train AUC=0.7533, validation AUC=0.7450
Epoch 09: train AUC=0.7538, validation AUC=0.7451
Epoch 10: train AUC=0.7552, validation AUC=0.7458
Epoch 11: train AUC=0.7564, validation AUC=0.7465
Epoch 12: train AUC=0.7566, validation AUC=0.7455
Epoch 13: train AUC=0.7573, validation AUC=0.7457
Epoch 14: train AUC=0.7589, validation AUC=0.7459
Epoch 15: train AUC=0.7594, validation AUC=0.7461
Best epoch: 11; validation AUC: 0.74649
F1-selected threshold: 14.4%


,protocol,partition,selected_epoch,threshold,n,difficulty_prevalence,majority_accuracy,auc,accuracy,precision,recall,f1,false_positive_rate,tn,fp,fn,tp
0,no_previous_application,train,11,0.144,232472,0.0819,0.9181,0.7564,0.8357,0.2344,0.4438,0.3068,0.1294,185813,27613,10593,8453
1,no_previous_application,blackbox,11,0.144,58118,0.0819,0.9181,0.7465,0.8342,0.2284,0.4306,0.2984,0.1298,46430,6927,2711,2050
2,no_previous_application,test,11,0.144,16421,0.0596,0.9404,0.7685,0.8849,0.2161,0.3548,0.2686,0.0815,14184,1259,631,347


## 4 Compare the questions and report the results

In [8]:
stratified_metrics=pd.read_json(main_dir/'final_metrics.json')
stratified_metrics['protocol']='stratified'
stratified_metrics['selected_epoch']=main_info['selected_epoch']
comparison=pd.concat([stratified_metrics,random_metrics,prior_metrics],ignore_index=True)
comparison.to_json(run_dir/'comparison.json',orient='records',indent=2)
display(comparison[['protocol','partition','n','difficulty_prevalence','auc','accuracy','precision','recall','f1','false_positive_rate','threshold']].round(4))
Path('active_final_split_run.json').write_text(json.dumps({'run_dir':str(run_dir)},indent=2))
print('Saved comparison:',run_dir)


Saved comparison: final_split_runs/20261013T182318505371Z


,protocol,partition,n,difficulty_prevalence,auc,accuracy,precision,recall,f1,false_positive_rate,threshold
0,stratified,train,184506,0.0807,0.7595,0.8378,0.2342,0.4448,0.3069,0.1277,0.121
1,stratified,blackbox,61002,0.0807,0.7468,0.8333,0.2241,0.4323,0.2951,0.1315,0.121
2,stratified,test,61503,0.0807,0.7495,0.8375,0.2303,0.4326,0.3006,0.1269,0.121
3,random,train,184506,0.0810,0.7648,0.8277,0.2305,0.4823,0.3120,0.1419,0.128
4,random,blackbox,61002,0.0795,0.7491,0.8255,0.2167,0.4573,0.2941,0.1427,0.128
5,random,test,61503,0.0811,0.7503,0.8211,0.2162,0.4591,0.2940,0.1470,0.128
6,no_previous_application,train,232472,0.0819,0.7564,0.8357,0.2344,0.4438,0.3068,0.1294,0.144
7,no_previous_application,blackbox,58118,0.0819,0.7465,0.8342,0.2284,0.4306,0.2984,0.1298,0.144
8,no_previous_application,test,16421,0.0596,0.7685,0.8849,0.2161,0.3548,0.2686,0.0815,0.144


Stratified remains the primary evaluation of applicants similar to the source population. The no-previous-application scenario assesses transfer to applicants without recorded prior applications. Interpret changes alongside outcome prevalence and the dev-selected threshold; higher AUC can coexist with lower recall in a new population. Random sampling is the required reference protocol, not a competing model choice.

Saved group metrics describe fairness within each evaluation population. Differences between protocols mix training composition and evaluation-population changes, so they cannot isolate discrimination or prove one split fairer. The 500 eyeball applicants were excluded from all alternate fits and metrics; individual review continues only for the main selected model.


## 5 Bureau availability within the previous application holdout

Keep the previous-application split, saved model and dev-selected threshold fixed. Within its test set (no recorded prior Home Credit applications), compare applicants with an eligible bureau record versus those without one. An eligible record has DAYS_CREDIT <= 0; missing or future credit dates do not establish application-time history.

Bureau availability is an analysis label only. No bureau feature enters the model, and no applicant changes partitions. Absence of a record is not proof of no credit history. This is exploratory analysis of an already evaluated holdout, not a new independent test or an invitation to tune on its results. Differences may reflect population composition and prevalence; they do not establish a causal effect or demographic fairness.

Run this section alone to analyze saved predictions without retraining. In Colab, upload bureau.csv and the saved final_split_runs folder, and set SAVED_SPLIT_RUN if needed.


In [9]:
from pathlib import Path
import json, hashlib
import numpy as np
import pandas as pd
from sklearn.metrics import roc_auc_score, accuracy_score, precision_score, recall_score, f1_score, confusion_matrix
from IPython.display import display

SAVED_SPLIT_RUN = None
if SAVED_SPLIT_RUN is not None:
    analysis_run=Path(SAVED_SPLIT_RUN)
else:
    pointer=Path('active_final_split_run.json')
    analysis_run=Path(json.loads(pointer.read_text())['run_dir'])
folder=analysis_run/'no_previous_application'
prediction_path=folder/'test_predictions.json'
before_hash=hashlib.sha256(prediction_path.read_bytes()).hexdigest()
pred=pd.read_json(prediction_path)
settings=json.loads((folder/'settings.json').read_text())
cutoff=float(settings['threshold'])
assert set(pred.SK_ID_CURR)==set(settings['split_ids']['test'])
assert pred.SK_ID_CURR.is_unique
paths=[Path('bureau.csv'),Path('/content/bureau.csv'),Path.home()/'Downloads'/'bureau.csv']
bureau_path=next((p for p in paths if p.exists()),None)
if bureau_path is None: raise FileNotFoundError('Provide bureau.csv beside this notebook, in Downloads, or in /content.')
bureau=pd.read_csv(bureau_path,usecols=['SK_ID_CURR','SK_ID_BUREAU','DAYS_CREDIT'])
assert bureau.SK_ID_BUREAU.is_unique
eligible=bureau.DAYS_CREDIT.le(0)
history_ids=set(bureau.loc[eligible,'SK_ID_CURR'])
pred['bureau_group']=np.where(pred.SK_ID_CURR.isin(history_ids),'Bureau record present','No eligible bureau record')

def wilson(k,n):
    if n==0: return (np.nan,np.nan)
    z=1.959963984540054;p=k/n;den=1+z*z/n
    center=(p+z*z/(2*n))/den;half=z*np.sqrt(p*(1-p)/n+z*z/(4*n*n))/den
    return max(0.,center-half),min(1.,center+half)

def summarize(group,frame):
    y=frame.actual_TARGET;score=frame.predicted_probability;label=score.ge(cutoff)
    tn,fp,fn,tp=confusion_matrix(y,label,labels=[0,1]).ravel()
    row=dict(group=group,n=len(frame),threshold=cutoff,positives=int(tp+fn),negatives=int(tn+fp),
        difficulty_prevalence=float(y.mean()),auc=roc_auc_score(y,score) if y.nunique()==2 else np.nan,
        accuracy=accuracy_score(y,label),precision=tp/(tp+fp) if tp+fp else np.nan,
        recall=tp/(tp+fn) if tp+fn else np.nan,f1=f1_score(y,label,zero_division=0),
        false_positive_rate=fp/(fp+tn) if fp+tn else np.nan,
        favorable_prediction_rate=float((~label).mean()),tn=int(tn),fp=int(fp),fn=int(fn),tp=int(tp))
    for name,k,total in [('recall',tp,tp+fn),('false_positive_rate',fp,fp+tn),('precision',tp,tp+fp)]:
        row[name+'_lower'],row[name+'_upper']=wilson(k,total)
    row['small_group_warning']=bool(min(tp+fn,tn+fp,tp+fp)<30)
    return row
summary=pd.DataFrame([summarize('All no-prior-application applicants',pred)]+[
    summarize(group,frame) for group,frame in pred.groupby('bureau_group')])
assert summary.iloc[1:].n.sum()==len(pred)
for count in ['tp','tn','fp','fn']:
    assert summary.iloc[1:][count].sum()==summary.iloc[0][count]
assert hashlib.sha256(prediction_path.read_bytes()).hexdigest()==before_hash
summary.to_csv(folder/'bureau_availability_metrics.csv',index=False)
summary.to_json(folder/'bureau_availability_metrics.json',orient='records',indent=2)
(folder/'bureau_analysis_provenance.json').write_text(json.dumps(dict(
    split_run=str(analysis_run),threshold=cutoff,prediction_sha256=before_hash,
    bureau_path=str(bureau_path),eligibility='DAYS_CREDIT <= 0',
    excluded_bureau_rows=int((~eligible).sum()),model_changed=False,split_changed=False,
    interpretation='Exploratory held-out subgroup analysis; no causal or demographic fairness claim'),indent=2))
print('Frozen threshold:',cutoff,'| Applicants:',len(pred),'| No retraining or changed predictions')
display(summary[['group','n','positives','difficulty_prevalence','auc','accuracy','precision','recall','f1','false_positive_rate','small_group_warning']].round(4))
display(summary[['group','recall_lower','recall_upper','false_positive_rate_lower','false_positive_rate_upper']].round(4))
print('Saved:',folder/'bureau_availability_metrics.csv')


Frozen threshold: 0.144 | Applicants: 16421 | No retraining or changed predictions
Saved: final_split_runs/20261013T182318505371Z/no_previous_application/bureau_availability_metrics.csv


,group,n,positives,difficulty_prevalence,auc,accuracy,precision,recall,f1,false_positive_rate,small_group_warning
0,All no-prior-application applicants,16421,978,0.0596,0.7685,0.8849,0.2161,0.3548,0.2686,0.0815,False
1,Bureau record present,13956,798,0.0572,0.7664,0.8841,0.2030,0.3509,0.2572,0.0835,False
2,No eligible bureau record,2465,180,0.0730,0.7794,0.8892,0.2952,0.3722,0.3292,0.0700,False


,group,recall_lower,recall_upper,false_positive_rate_lower,false_positive_rate_upper
0,All no-prior-application applicants,0.3254,0.3853,0.0773,0.0859
1,Bureau record present,0.3186,0.3846,0.0789,0.0884
2,No eligible bureau record,0.3050,0.4448,0.0603,0.0812
